# VeriStream: Modular Deepfake Forensic Notebook
This notebook imports directly from your modular `veristream/` package so each layer stays clean and easy to debug:
- `veristream/config.py` — Paths, thresholds, and weights
- `veristream/metadata.py` — Layer 0: C2PA & EXIF Pre-Check
- `veristream/router.py` — Layer 1: MTCNN Smart Router (`face_margin=0.50`)
- `veristream/global_stream.py` — Layer 2: Swin V2 + GAR, Depth Anything V2, MediaPipe Pose
- `veristream/local_stream.py` — Layer 3: SRM, CrossViT + GAR, FaceMesh, EAR, FFT, Mesh-GAR Overlap, Face X-ray/Skin
- `veristream/pipeline.py` — Layer 4: Late Fusion Orchestrator
- `veristream/gemini_engine.py` — Layer 5: Gemini Forensic Narrative

In [ ]:
from pathlib import Path
import json
import matplotlib.pyplot as plt
from veristream import Config, VeriStream, DEVICE

cfg = Config(face_margin=0.50, use_faceguard=False)
print(f'Active Device         : {DEVICE}')
print(f'Swin V2 Checkpoint    : {cfg.swin_ckpt} (Exists: {cfg.swin_ckpt.exists()})')
print(f'CrossViT Checkpoint   : {cfg.crossvit_ckpt} (Exists: {cfg.crossvit_ckpt.exists()})')

In [ ]:
# Initialize the modular orchestrator
vs = VeriStream(cfg)

target_image = input('Enter full path to an image to analyze: ').strip().strip("'\"")
if Path(target_image).exists():
    rep, vis = vs.analyze(target_image)
    print('\n' + '='*65)
    print(f"VERDICT: {rep['verdict']} | Manipulation Probability: {rep['confidence_percentage']}")
    print('='*65)
    for e in rep['evidence']:
        print(f"  [{e['stream'].upper():6s}] {e['name']:36s} -> score={e['score']:.3f} ({e['note']})")
    if rep.get('module_warnings'):
        print('\nModule Warnings:', rep['module_warnings'])
    
    if vis:
        cols = min(len(vis), 4)
        rows = (len(vis) + cols - 1) // cols
        fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 4 * rows))
        axes_flat = axes.flatten() if hasattr(axes, 'flatten') else [axes]
        for ax, (k, arr) in zip(axes_flat, vis.items()):
            ax.imshow(arr)
            ax.set_title(k.replace('_', ' ').upper(), fontsize=10, fontweight='bold')
            ax.axis('off')
        for ax in axes_flat[len(vis):]:
            ax.axis('off')
        plt.tight_layout()
        plt.show()
else:
    print(f'File not found: {target_image}')